In [ ]:
from pyspark.sql.functions import col, sum as _sum, count, round
silver_df = spark.read.table("ecommerce_silver")

In [ ]:
# Calculate Business Metrics
customer_metrics_df = silver_df.groupBy("CustomerID", "Country").agg(
    round(_sum(col("Quantity") * col("UnitPrice")), 2).alias("TotalSpent"),
    count("InvoiceNo").alias("TransactionCount")
)

In [ ]:
# Save to Gold Table and Optimize
(customer_metrics_df.write.format("delta")
  .mode("overwrite")
  .option("mergeSchema", "true")
  .saveAsTable("gold_customer_metrics"))

spark.sql("OPTIMIZE gold_customer_metrics ZORDER BY (Country)")

print("Gold Layer Update: SUCCESS")
display(customer_metrics_df.limit(10))